MD# Caminho 1: Python no Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/souuzaa/gdg-blumenau-ml-gcp/blob/main/01-local-e-colab/penguins_sklearn.ipynb)

Talk *Do notebook à nuvem*, GDG Blumenau. Vamos descobrir a espécie de um pinguim (Adelie, Chinstrap ou Gentoo) só pelas medidas do corpo.

Rode as células em ordem com **Shift+Enter**.

MD## 1. Carregar os dados

In [ ]:
import pandas as pd
from pathlib import Path

CSV_URL = "https://raw.githubusercontent.com/souuzaa/gdg-blumenau-ml-gcp/main/data/penguins.csv"
local = Path("../data/penguins.csv")
df = pd.read_csv(local if local.exists() else CSV_URL)

# Igual ao slide: remove qualquer linha com valor vazio.
df = df.dropna()
print(len(df), "pinguins")
df.head()

MD## 2. Olhar antes de treinar

Quantos de cada espécie? Dá para separar no olho?

In [ ]:
import matplotlib.pyplot as plt

print(df["species"].value_counts(), "\n")
fig, ax = plt.subplots(figsize=(7, 5))
for especie, grupo in df.groupby("species"):
    ax.scatter(grupo["bill_length_mm"], grupo["flipper_length_mm"], label=especie, alpha=0.7)
ax.set_xlabel("comprimento do bico (mm)")
ax.set_ylabel("nadadeira (mm)")
ax.legend()
plt.show()

MD## 3. Separar treino e teste

20% dos pinguins ficam guardados. O modelo nunca vê esses durante o treino: é a prova com questões que o aluno não viu na lista de exercício.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X, y = df[FEATURES], df["species"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"treino: {len(X_tr)} | teste: {len(X_te)}")

MD## 4. Treinar

`StandardScaler` coloca milímetro e grama na mesma escala. `LogisticRegression` aprende a fronteira entre as espécies.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression())
model.fit(X_tr, y_tr)
print(model.score(X_te, y_te))

MD## 5. Avaliar com calma

100% deveria te deixar desconfiado. Em dado real, quase sempre é vazamento de dado ou bug. Aqui o problema é fácil de verdade, mas vamos conferir de dois jeitos.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score

print(classification_report(y_te, model.predict(X_te)))
ConfusionMatrixDisplay.from_estimator(model, X_te, y_te)
plt.show()

# Validação cruzada: repete a prova em 5 divisões diferentes dos dados.
cv = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression()), X, y, cv=5)
print(f"Validação cruzada: média {cv.mean():.1%}, pior divisão {cv.min():.1%}")

MD## 6. Um pinguim novo

Mude os números e veja a previsão mudar.

In [ ]:
novo = pd.DataFrame([[45.0, 15.5, 215.0, 5000.0]], columns=FEATURES)
print("previsão:", model.predict(novo)[0])
{c: round(float(p), 3) for c, p in zip(model.classes_, model.predict_proba(novo)[0])}

MD## Experimentos

1. Treine com só duas medidas. Quanto a acurácia cai?
2. Troque `LogisticRegression` por `RandomForestClassifier` ou `KNeighborsClassifier`.
3. Tire o `StandardScaler`. De onde vem o aviso de convergência?

Próximo passo: o mesmo modelo em SQL, no [BigQuery ML](https://github.com/souuzaa/gdg-blumenau-ml-gcp/tree/main/02-bigquery-ml).